# Artwork Multi-Method Eval on Kaggle

This notebook runs artwork evaluation on Kaggle with three methods:

- `KVzip`
- `Finch_Full`
- `Finch_CPT`

It supports checkpoint resume, strict image checks, smoke tests, and CE-style evaluation output.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"

WORK_DIR = Path("/kaggle/working")
BENCH_DIR = WORK_DIR / "kv-compression-benchmark"
CE_DIR = WORK_DIR / "CompressionExperiments"

gh_token = (os.environ.get("GITHUB_TOKEN") or "").strip() or None

def clone_public_or_token(url: str, out_dir: Path, token: str | None, repo_name: str) -> bool:
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(out_dir)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"{repo_name}: public clone OK")
        return True
    print(f"{repo_name}: public clone failed (code={r.returncode})")
    if r.stderr:
        print((r.stderr or "")[:500])
    if not token:
        return False
    private_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r2 = subprocess.run(["git", "clone", "--depth", "1", private_url, str(out_dir)], capture_output=True, text=True)
    if r2.returncode == 0:
        print(f"{repo_name}: token clone OK")
        return True
    print(f"{repo_name}: token clone failed (code={r2.returncode})")
    if r2.stderr:
        print((r2.stderr or "")[:500])
    return False

os.chdir(str(WORK_DIR))
for p in [BENCH_DIR, CE_DIR]:
    if p.exists():
        shutil.rmtree(p)

bench_ok = clone_public_or_token(BENCH_URL, BENCH_DIR, gh_token, "BENCH")
ce_ok = clone_public_or_token(CE_URL, CE_DIR, gh_token, "CE")
if not ce_ok:
    raise RuntimeError("Failed to clone CompressionExperiments")
print("Clone step finished. BENCH available:", bench_ok)

In [ ]:
import inspect
import subprocess
import sys
from pathlib import Path

os.chdir("/kaggle/working/CompressionExperiments")
subprocess.check_call(["git", "submodule", "update", "--init", "--recursive"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45.0", "accelerate>=0.30.0", "bitsandbytes>=0.43.0",
    "pandas", "pyyaml", "scikit-learn", "matplotlib", "tqdm", "pillow"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "kvpress"])

import kvpress
finch_path = Path(inspect.getfile(kvpress.FinchPress))
txt = finch_path.read_text(encoding="utf-8")
txt_new = txt
if "self.hook = None" not in txt_new and "def __enter__(self, model):" in txt_new:
    txt_new = txt_new.replace("def __enter__(self, model):", "def __enter__(self, model):\n        self.hook = None")
txt_new = txt_new.replace("hook.remove()", "if hasattr(self, 'hook') and self.hook: self.hook.remove()")
if txt_new != txt:
    finch_path.write_text(txt_new, encoding="utf-8")
    print("Applied Finch hotfix:", finch_path)
print("Install complete.")

In [ ]:
from pathlib import Path
import shutil

SRC_BENCH_ART = Path("/kaggle/working/kv-compression-benchmark/datasets/artwork")
SRC_BENCH_CSV = SRC_BENCH_ART / "paintings.csv"
SRC_BENCH_IMG = SRC_BENCH_ART / "images"

DST_ART = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork")
DST_ART.mkdir(parents=True, exist_ok=True)

if SRC_BENCH_CSV.is_file():
    shutil.copy2(SRC_BENCH_CSV, DST_ART / "paintings.csv")
    print("CSV source: benchmark repo")
elif (DST_ART / "paintings.csv").is_file():
    print("CSV source: CE built-in")
else:
    raise FileNotFoundError("No paintings.csv found")

ce_images = DST_ART / "images"
if ce_images.is_dir() and any(ce_images.iterdir()):
    print("Images source: CE built-in")
elif SRC_BENCH_IMG.is_dir() and any(SRC_BENCH_IMG.iterdir()):
    if ce_images.exists():
        shutil.rmtree(ce_images)
    shutil.copytree(SRC_BENCH_IMG, ce_images)
    print("Images source: benchmark repo")
else:
    raise FileNotFoundError("No usable images found")

print("Dataset ready:", DST_ART)
print("image count:", len(list(ce_images.glob("*"))))

In [ ]:
import pandas as pd
from pathlib import Path
from urllib.parse import unquote
from PIL import Image, ImageFile

DATASET_PATH = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/paintings.csv")
IMAGES_DIR = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/images")
MAX_ROWS = 10

ImageFile.LOAD_TRUNCATED_IMAGES = False
df_check = pd.read_csv(DATASET_PATH).head(MAX_ROWS).copy()
bad_rows = []
for i, row in df_check.iterrows():
    tail = str(row["image_url"]).split("/")[-1].split("?")[0]
    p_un = IMAGES_DIR / unquote(tail)
    p_raw = IMAGES_DIR / tail
    target = p_un if p_un.is_file() else p_raw
    try:
        with Image.open(target) as im:
            _ = im.convert("RGB")
        print(f"OK Row {i}: {target.name}")
    except Exception as e:
        bad_rows.append((i, str(target), str(e)[:120]))
        print(f"BAD Row {i}: {target.name} -> {str(e)[:120]}")
if bad_rows:
    raise RuntimeError(f"Bad images found before model load: {bad_rows}")
print("Strict image check passed.")

In [ ]:
import importlib.util
import torch
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True)

processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    quantization_config=bnb_cfg,
)

_patch = "/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py"
_spec = importlib.util.spec_from_file_location("patch", _patch)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)
print("Model ready.")

In [ ]:
import yaml
import pandas as pd
from urllib.parse import unquote
from pathlib import Path
from kvpress import KVzipPress, FinchPress

MAX_ROWS = 10
MAX_NEW_TOKENS = 50
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]

OUTPUT_ROOT = Path("/kaggle/working/ce_artwork_all_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_ROOT / "checkpoint_all.csv"
RESULTS_ROOT = OUTPUT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIGS = {
    "KVzip": (KVzipPress, True),
    "Finch_Full": (FinchPress, False),
    "Finch_CPT": (FinchPress, True),
}

QCFG = Path("/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/configs/image_queries.yaml")
with QCFG.open("r", encoding="utf-8") as f:
    q_cfg = yaml.safe_load(f)["artwork"]
queries_plan = [(q, True) for q in q_cfg["filter_queries"]] + [(q, False) for q in q_cfg["extract_queries"]]

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS).copy()
df["record_id"] = range(len(df))
def resolve_image(url: str) -> str:
    tail = str(url).split("/")[-1].split("?")[0]
    p = IMAGES_DIR / unquote(tail)
    return str(p) if p.is_file() else str(IMAGES_DIR / tail)
df["image_path"] = df["image_url"].apply(resolve_image)

_EXTRACT_SUFFIX = (" (be concise, no explanation, no introductory text, just the answer,"
                   " output datatype: STRING, do not repeat the datatype in the answer) ?")
def build_answer_prefix(row, question: str, is_boolean: bool, use_cpt: bool) -> str:
    context = "" if use_cpt else f"This painting is titled '{row['title']}', created around {row['inception']}. It belongs to the {row['movement']} movement and {row['genre']} genre. "
    if is_boolean:
        return (f"{context}Answer the following question based on the image with '1' or '0'. Do not add any other comments. {question}\nAnswer: ")
    return (f"{context}Answer the following question based on the image. Do not add any other comments. {question + _EXTRACT_SUFFIX}\nAnswer: ")
print("Configuration ready.")

In [ ]:
import time
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

SMOKE_ROWS = min(3, len(df))
SMOKE_METHODS = ["KVzip", "Finch_Full", "Finch_CPT"]
SMOKE_RATIO = 0.4
qtext, is_bool = queries_plan[0]
_tok = getattr(processor, "tokenizer", processor)

def _run_once(image_path, answer_prefix, PressCls, ratio):
    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, max_new_tokens=24, pad_token_id=_tok.pad_token_id)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

results = []
for i in range(SMOKE_ROWS):
    row = df.iloc[i]
    rid = int(row["record_id"])
    img_path = row["image_path"]
    with Image.open(img_path) as im:
        _ = im.convert("RGB")
    for m_name in SMOKE_METHODS:
        Cls, use_cpt = EXPERIMENT_CONFIGS[m_name]
        ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
        t0 = time.perf_counter()
        ok, err, out = True, "", ""
        try:
            out = _run_once(img_path, ap, Cls, SMOKE_RATIO)
        except Exception as e:
            ok = False
            err = str(e)[:200]
        dt = (time.perf_counter() - t0) * 1000
        results.append((rid, m_name, ok, dt, out[:100], err))

for rid, m_name, ok, dt, out_head, err_head in results:
    if ok:
        print(f"PASS rid={rid} method={m_name} t={dt:.1f}ms out={out_head!r}")
    else:
        print(f"FAIL rid={rid} method={m_name} t={dt:.1f}ms err={err_head!r}")
if any(not r[2] for r in results):
    raise RuntimeError("Smoke test failed. Fix before full run.")
print("Smoke test passed.")

In [ ]:
import csv as _csv
import gc
import logging
import os
import pandas as pd
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
logging.getLogger("kvpress.presses.kvzip_press").setLevel(logging.ERROR)

# Finch runtime safety patch
_orig_enter = getattr(FinchPress, "__enter__", None)
_orig_exit = getattr(FinchPress, "__exit__", None)
def _safe_finch_enter(self, model):
    self.hook = None
    return _orig_enter(self, model) if _orig_enter is not None else self
def _safe_finch_exit(self, exc_type, exc_value, traceback):
    h = getattr(self, "hook", None)
    if h is not None:
        try:
            h.remove()
        except Exception:
            pass
    try:
        return _orig_exit(self, exc_type, exc_value, traceback) if _orig_exit is not None else False
    except UnboundLocalError:
        return False
FinchPress.__enter__ = _safe_finch_enter
FinchPress.__exit__ = _safe_finch_exit

_tok = getattr(processor, "tokenizer", processor)
def run_generate_vision(image_path, answer_prefix, PressCls, ratio):
    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv = [{"role":"user","content":[{"type":"image"},{"type":"text","text":answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=_tok.pad_token_id)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def load_done(path):
    if not path.exists():
        return set()
    ck = pd.read_csv(path)
    done = set()
    for _, r in ck.iterrows():
        if str(r.get("error", "")).strip().lower() in ("", "nan"):
            done.add((str(r["method"]), str(float(r["ratio"])), int(r["record_id"]), str(r["query"])))
    return done

done_keys = load_done(CHECKPOINT_PATH)
print(f"Resuming: {len(done_keys)} already done.")
f = CHECKPOINT_PATH.open("a", newline="", encoding="utf-8")
writer = _csv.DictWriter(f, fieldnames=["method", "ratio", "record_id", "query", "answer", "error"])
if CHECKPOINT_PATH.stat().st_size == 0:
    writer.writeheader()

for m_name, (Cls, use_cpt) in EXPERIMENT_CONFIGS.items():
    print(f"\\n>>> Starting Method: {m_name}")
    for ratio in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            rid = int(row["record_id"])
            for qtext, is_bool in queries_plan:
                key = (m_name, str(float(ratio)), rid, qtext)
                if key in done_keys:
                    continue
                ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
                ans = ""
                err = ""
                try:
                    ans = run_generate_vision(row["image_path"], ap, Cls, ratio)
                except Exception as e:
                    err = str(e)[:500]
                    print(f"ERROR [rid={rid} {m_name} R{ratio}]: {err[:120]}")
                writer.writerow({"method": m_name, "ratio": ratio, "record_id": rid, "query": qtext, "answer": ans, "error": err})
                f.flush()
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
f.close()
print("Inference finished.")

In [ ]:
import pandas as pd
import sys
from pathlib import Path

ck_all = pd.read_csv(CHECKPOINT_PATH)
ea_path = Path("/kaggle/working/evaluation_results.csv")

ok = ck_all[ck_all.error.fillna("").astype(str).str.strip().isin(["", "nan"])].copy()
for (m, r), grp in ok.groupby(["method", "ratio"]):
    d = RESULTS_ROOT / "artwork" / f"Llava3-8b-{m}" / f"{float(r):.2f}"
    d.mkdir(parents=True, exist_ok=True)
    grp[["record_id", "query", "answer"]].to_csv(d / "results.csv", index=False)

sys.path.append("/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval")
from evaluation.evaluator import EvaluationManager
mgr = EvaluationManager(
    config_path="/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/evaluation/evaluation_config.yaml",
    results_dir=str(RESULTS_ROOT),
)
res = mgr.evaluate_all()
out_csv = OUTPUT_ROOT / "evaluation_results_all_methods.csv"
res.to_csv(out_csv, index=False)
print("Wrote:", out_csv)
display(res.groupby(["model_tag", "ratio"])[["precision", "recall", "f1"]].mean())